In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
    if (!require(pkg, character.only = TRUE)) {
        install.packages(pkg)
        library(pkg, character.only = TRUE)
    }
}

library(CCA)        # funkcje CCA i cc()
library(yacca)      # alternatywne funkcje CCA
library(tidyverse)  # wczytywanie danych, dplyr, ggplot2
library(ggplot2)
library(assertthat)

In [ ]:
# Chunk 2: Wczytanie danych butterfly
url <- "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/butterfly.dat"
butterfly <- read.table(url, header = TRUE)

# W/g skryptu: zmienna a = allele5 + allele6
a <- butterfly[,5] + butterfly[,6]

# Zestaw Y = a + kolumny 7,8,9
Y <- cbind(a, butterfly[,7:9])

# Zestaw X = kolumny 1:4
X <- butterfly[,1:4]

# Konwersja do macierzy
X <- as.matrix(X)
Y <- as.matrix(Y)

# Canonical Correlation Analysis:
canon <- cca(
  X, Y,
  xcenter = TRUE, ycenter = TRUE,
  xscale  = TRUE, yscale  = TRUE,
  standardize.scores = TRUE
)

canon$corr      # korelacje kanocnzie 
canon$chisq     # statystyka testowe chi² (same wartości) 
canon$df        # liczba stopni swobody dla danego testu 
pchisq(canon$chisq, canon$df, lower.tail = FALSE) # to jest 1 - chisq 


In [ ]:
# Chunk 3: Pierwsze pary zmiennych kanonicznych

u1 <- X %*% canon$xcoef[,1]
v1 <- Y %*% canon$ycoef[,1]

# Wykres zależności
qplot(u1, v1) +
  labs(title="Pierwsza para zmiennych kanonicznych: Butterfly",
       x="u1 (X-side)", y="v1 (Y-side)")


Zadanie 2

refleksje 
- szukamy zmiennych które dają największą korelacje pomiędzy **nowymi** cechami, co pozwala nam potem powiązać je z **klasami** 
    - czy możemy to wykorzystać do tworzenia lepszych cech klasyfikacyjnych, na zasadzie, robię np SVM albo coś takiego na danych cechach to teraz przy inżynierii zmiennych możemy otrzymać lepsze dopasowanie, ponieważ tworzymy wcześniej cechy z odpowiednią kombinacją poprzednich zmiennych żeby nam korelowały między sobą, jak znajdziemy korelacje między dwiema zmiennymi, to wtedy możemy to wykorzstać bo te dwie cechy będą istotnie charakteryzowały daną cechę 
    - jak istotne są te testy które to porównują. 

In [ ]:
# Chunk 4: Wczytanie danych Penguin
penguins <- read_csv(
  "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/palmer_penguins.csv"
)

penguins <- penguins %>% drop_na()

# Zestawy danych dla CCA
X <- penguins %>%
  select(bill_depth_mm, bill_length_mm) %>%
  scale()

Y <- penguins %>%
  select(flipper_length_mm, body_mass_g) %>%
  scale()


In [ ]:
# Chunk 5: CCA dla pingwinów

cc_results <- cancor(X, Y)

cc_results$xcoef   # współczynniki dla X
cc_results$ycoef   # współczynniki dla Y
cc_results$cor     # korelacje kanoniczne - między tymi zmiennymi kanonicznymi 


In [ ]:
# Chunk 6: Wyliczenie pierwszej i drugiej pary zmiennych kanonicznych

CC1_X <- as.matrix(X) %*% cc_results$xcoef[,1]
CC1_Y <- as.matrix(Y) %*% cc_results$ycoef[,1]

CC2_X <- as.matrix(X) %*% cc_results$xcoef[,2]
CC2_Y <- as.matrix(Y) %*% cc_results$ycoef[,2]

# Weryfikacja korelacji
cor(CC1_X, CC1_Y)
assertthat::are_equal(cc_results$cor[1], cor(CC1_X, CC1_Y)[1])


In [ ]:
cca_df <- penguins %>%
  mutate(
    CC1_X = CC1_X,
    CC1_Y = CC1_Y,
    CC2_X = CC2_X,
    CC2_Y = CC2_Y
  )

# Wykres: pierwsza para zmiennych kanonicznych
cca_df %>%
  ggplot(aes(CC1_X, CC1_Y)) +
  geom_point() +
  labs(title="Pingwiny — pierwsza para zmiennych kanonicznych")

# Boxplot CC1_X ~ species
cca_df %>%
  ggplot(aes(species, CC1_X, color = species)) +
  geom_boxplot(width=0.5) +
  geom_jitter(width=0.15) +
  theme(legend.position="none")

# Boxplot CC1_Y ~ species
cca_df %>%
  ggplot(aes(species, CC1_Y, color = species)) +
  geom_boxplot(width=0.5) +
  geom_jitter(width=0.15)

# Scatter colored by species
cca_df %>%
  ggplot(aes(CC1_X, CC1_Y, color = species)) +
  geom_point()


In [ ]:
cca_df %>%
  ggplot(aes(CC2_X, CC2_Y, color = sex)) +
  geom_point() +
  labs(title="Druga para zmiennych kanonicznych — płeć")


Exercise 3

In [ ]:
# Chunk 9: Wczytanie danych — psychologiczne i akademickie
mm <- read.csv("https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/mmreg.csv")

colnames(mm) <- c("Control", "Concept", "Motivation",
                  "Read", "Write", "Math", "Science", "Sex")

psych <- mm[,1:3]
acad  <- mm[,4:8]

matcor(psych, acad)


In [ ]:
cc1 <- cc(psych, acad)

cc1$cor        # korelacje kanoniczne
cc1$xcoef      # współczynniki X
cc1$ycoef      # współczynniki Y

# Testy istotności wg skryptu
n <- nrow(psych)
p <- ncol(psych)
q <- ncol(acad)

ev <- (1 - cc1$cor^2)
k  <- min(p, q)
m  <- n - 3/2 - (p + q)/2
w  <- rev(cumprod(rev(ev)))

d1 <- d2 <- f <- numeric(k)

for (i in 1:k) {
  s  <- sqrt((p^2 * q^2 - 4)/(p^2 + q^2 - 5))
  si <- 1/s
  d1[i] <- p*q
  d2[i] <- m*s - p*q/2 + 1
  r <- (1 - w[i]^si) / w[i]^si
  f[i] <- r * d2[i] / d1[i]
  
  p <- p - 1
  q <- q - 1
}

pv <- pf(f, d1, d2, lower.tail = FALSE)

# Tabela wyników
dmat <- cbind(WilksL = w, F = f, df1 = d1, df2 = d2, p = pv)
dmat


In [ ]:
# Chunk 11: Standaryzowane współczynniki

s1 <- diag(sqrt(diag(cov(psych))))
s2 <- diag(sqrt(diag(cov(acad))))

std_xcoef <- s1 %*% cc1$xcoef
std_ycoef <- s2 %*% cc1$ycoef

std_xcoef
std_ycoef


dopisać
- do testów: rozróżnić asymptotyczne i zwykłe (bo zwykłę są zbieżne a asymptotyczne dopiero w końcu zbiegają) 